In [1]:
%pip install openai scikit-learn pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Create client

from openai import OpenAI
from sklearn.model_selection import train_test_split
import pandas as pd

client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",  # Ollama
    api_key="sk-1234"  # Dummy key
)

In [3]:
# Set model parameters

SYSTEM_PROMPT = """
You are an ACE (Adverse Childhood Experiences) survey scoring assistant.
Your job is to extract answers for the ACE questionnaire based on a given conversation transcript.
The ACE questionnaire assesses 10 categories of adverse childhood experiences before age 18.

ACE question order:
q1:  Did a parent or other adult in the household often or very often swear at you, insult you, put you down, or humiliate you? Or act in a way that made you afraid you might be physically hurt?
q2:  Did a parent or other adult in the household often or very often push, grab, slap, or throw something at you? Or ever hit you so hard that you had marks or were injured?
q3:  Did an adult or person at least 5 years older than you ever touch or fondle you or have you touch their body in a sexual way? Or attempt or actually have oral, anal, or vaginal intercourse with you?
q4:  Did you often or very often feel that no one in your family loved you or thought you were important or special? Or your family didn't look out for each other, feel close to each other, or support each other?
q5:  Did you often or very often feel that you didn't have enough to eat, had to wear dirty clothes, and had no one to protect you? Or your parents were too drunk or high to take care of you or take you to the doctor if you needed it?
q6:  Were your parents ever separated or divorced?
q7:  Was your mother or stepmother often or very often pushed, grabbed, slapped, or had something thrown at her? Or sometimes, often, or very often kicked, bitten, hit with a fist, or hit with something hard? Or ever repeatedly hit over at least a few minutes or threatened with a gun or knife?
q8:  Did you live with anyone who was a problem drinker or alcoholic, or who used street drugs?
q9:  Was a household member depressed or mentally ill, or did a household member attempt suicide?
q10: Did a household member go to prison?

Scale mapping:
0 = No
1 = Yes

Input Format:
You will take input in the form of a transcript in JSON matching OpenAI's API format.
Example:
[
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    // ...
]

Output Format:
The answer for each question must be a single string value: "0" (No) or "1" (Yes).
Rules:
- For each question q1-q10, determine the score based on the user's responses.
- Return ONLY valid JSON.
- Do not include markdown fences.
- Do not include explanation.
- Do not omit any q1-q10 key.
- expected_output must contain exactly q1 through q10.
- conversation must be consistent with expected_output.
Return JSON in exactly this shape:
{
  "q1": "0",
  "q2": "0",
  "q3": "0",
  "q4": "0",
  "q5": "0",
  "q6": "0",
  "q7": "0",
  "q8": "0",
  "q9": "0",
  "q10": "0"
}
"""
MODEL = 'qwen3:8b'

- Use a json dataset of ACE conversation transcripts
- Extract scores using LLM
- Validate whether they were correct using MSE

In [4]:
# Load the ACE dataset
import json

with open('ace_valid_dataset.json') as f:
    data = json.loads('\n'.join([line for line in f]))

# Split the data into conversation data and expected values
conversations = [entry['conversation'] for entry in data]
expected_outputs = [entry['expected_output'] for entry in data]

print('SUCCESS: Number of conversations matches number of scores' if len(conversations) == len(expected_outputs) else 'FAIL: Number of conversations does not match number of scores')
print(f'Total conversations: {len(conversations)}')

SUCCESS: Number of conversations matches number of scores
Total conversations: 10


In [5]:
# Further split the data into training and test data
conv_train, _conv_test, score_train, _score_test = train_test_split(conversations, expected_outputs, random_state=1234)
print(f'Train: {len(conv_train)}, Test: {len(_conv_test)}')

Train: 7, Test: 3


In [6]:
def Message(content, role='user'):
    return {
        'role': role,
        'content': content
    }

def create_prediction(transcript) -> str:
    completion = client.chat.completions.create(
        model=MODEL,
        temperature=0.1,  # low temperature for consistent scoring
        messages=[
            Message(SYSTEM_PROMPT, 'system'),
            Message(json.dumps(transcript))
        ],
    )
    return completion.choices[0].message.content

In [7]:
# Run predictions on training set
scores = []
for i, c in enumerate(conv_train):
    print(f'Generating {i+1}/{len(conv_train)}')
    scores.append(create_prediction(c))
    print(scores[-1])

Generating 1/7
{
  "q1": "0",
  "q2": "0",
  "q3": "0",
  "q4": "1",
  "q5": "0",
  "q6": "1",
  "q7": "0",
  "q8": "1",
  "q9": "1",
  "q10": "0"
}
Generating 2/7
{
  "q1": "0",
  "q2": "0",
  "q3": "0",
  "q4": "0",
  "q5": "0",
  "q6": "1",
  "q7": "0",
  "q8": "0",
  "q9": "0",
  "q10": "0"
}
Generating 3/7
{
  "q1": "1",
  "q2": "1",
  "q3": "0",
  "q4": "1",
  "q5": "0",
  "q6": "0",
  "q7": "0",
  "q8": "1",
  "q9": "1",
  "q10": "0"
}
Generating 4/7
{
  "q1": "1",
  "q2": "1",
  "q3": "1",
  "q4": "1",
  "q5": "1",
  "q6": "1",
  "q7": "1",
  "q8": "1",
  "q9": "1",
  "q10": "1"
}
Generating 5/7
{
  "q1": "0",
  "q2": "0",
  "q3": "0",
  "q4": "1",
  "q5": "0",
  "q6": "1",
  "q7": "0",
  "q8": "0",
  "q9": "1",
  "q10": "0"
}
Generating 6/7
{
  "q1": "1",
  "q2": "0",
  "q3": "0",
  "q4": "1",
  "q5": "1",
  "q6": "1",
  "q7": "0",
  "q8": "1",
  "q9": "1",
  "q10": "0"
}
Generating 7/7
{
  "q1": "1",
  "q2": "1",
  "q3": "0",
  "q4": "1",
  "q5": "1",
  "q6": "1",
  "q7": "1"

In [8]:
# Parse JSON scores
json_scores = []
for s in scores:
    try:
        json_scores.append(json.loads(s))
    except json.JSONDecodeError:
        print(f'Failed to parse: {s}')
        json_scores.append({f'q{i}': '0' for i in range(1, 11)})

In [9]:
from sklearn.metrics import mean_squared_error

def convert_scores_to_array(scores_df):
    return [int(s) if s else 0 for s in scores_df.values.flatten().tolist()]

pred_df = pd.DataFrame(json_scores)
train_df = pd.DataFrame(score_train)

# Ensure column order matches
cols = [f'q{i}' for i in range(1, 11)]
pred_df = pred_df[cols]
train_df = train_df[cols]

mse = mean_squared_error(
    convert_scores_to_array(train_df),
    convert_scores_to_array(pred_df)
)
print(f'MSE: {mse}')

diff_df = train_df.compare(pred_df, align_axis=1, keep_shape=True, keep_equal=True).rename(
    columns={'self': 'Expected', 'other': 'Predicted'}, level=1
)
display(diff_df)

MSE: 0.014285714285714285


q1                 q2                 q3                 q4            \
  Expected Predicted Expected Predicted Expected Predicted Expected Predicted   
0        0         0        0         0        0         0        1         1   
1        0         0        0         0        0         0        0         0   
2        0         1        1         1        0         0        1         1   
3        1         1        1         1        1         1        1         1   
4        0         0        0         0        0         0        1         1   
5        1         1        0         0        0         0        1         1   
6        1         1        1         1        0         0        1         1   

        q5                 q6                 q7                 q8            \
  Expected Predicted Expected Predicted Expected Predicted Expected Predicted   
0        0         0        1         1        0         0        1         1   
1        0         0        1         1        0         0        0         0   
2        0         0        0         0        0         0        1         1   
3        1         1        1         1        1         1        1         1   
4        0         0        1         1        0         0        0         0   
5        1         1        1         1        0         0        1         1   
6        1         1        1         1        1         1        1         1   

        q9                q10            
  Expected Predicted Expected Predicted  
0        1         1        0         0  
1        0         0        0         0  
2        1         1        0         0  
3        1         1        1         1  
4        1         1        0         0  
5        1         1        0         0  
6        1         1        1         1